# Column guide: the four Équinoxe files with our names

The Yardi column names (`sRent`, `hUnit`, `sTermSeq`…) are short and sometimes misleading. This notebook shows each of the
four files the way we work with it: every original column next to the name we gave it, what it really contains, and whether
we use it. It is a companion to `equinoxe_2026.ipynb` (§0.4 renames, §2.9 full dictionary): same renaming, same dictionary,
laid out file by file.

**Two rules we follow here:**
- The source CSV files are never modified; the renaming happens in code (brief: "les fichiers sources restent intacts").
- Each file is shown the simple way first: read with `pd.read_csv`, columns renamed, first 3 rows (the organisers' starter
  notebook shows rows the same way). The table under it explains every column. The data itself stays private: never
  published, never in a public repository.

**Reading the original names:** a column starting with **h** is a *handle*, an internal ID used only to join tables
(`hUnit` → `unit_id`). A column starting with **s** is a *stored value*, something a person reads or groups by
(`sBuilding` → `building`). The prefix says how the column is used, not its type.

In [ ]:
%cd nb
%run ./02_data_audit.ipynb
%cd ..

## How we build each file's guide
**What we do:** for every column of a file, in the file's own order, we look up our name and meaning in the dictionary of §2.9
(`COLUMN_MEANINGS`), then compute three facts from the data: type, number of distinct values, and a short summary (the
categories when there are few, otherwise the range). Identifiers, addresses, coordinates and links are summarised by their
count only.

In [ ]:
DATE_PATTERN = r"\d{4}-\d{2}(-\d{2})?"   # how dates are written in the CSV files
ROWS_PREVIEWED = 3                # same as leases.head(3) in the organisers' starter notebook
MAX_CATEGORIES_SHOWN = 8          # list the values only when a column has this many or fewer
NOT_SUMMARISED = {"unit_id", "property_id", "building_id", "unit_code", "unit_link", "address", "latitude", "longitude",
                  "unit_description"}   # identifiers and location details: count only
meanings = {original: (new, meaning, use) for original, new, meaning, use in COLUMN_MEANINGS}
RAW_FILES = {"listings": (raw_listings, LISTING_RENAME), "lease_history": (raw_leases, LEASE_RENAME),
             "concessions": (raw_concessions, CONCESSION_RENAME), "asking_history": (raw_asking, ASKING_RENAME)}


def summarise(series, new_name):
    """Short, row-free description of a column: categories, date range or numeric range."""
    values = series.dropna()
    if new_name in NOT_SUMMARISED:
        return ""
    if values.nunique() <= MAX_CATEGORIES_SHOWN:
        return ", ".join(sorted(map(str, values.unique())))
    if pd.api.types.is_numeric_dtype(values):
        return f"{values.min():,.0f} to {values.max():,.0f}"
    text = values.astype(str)
    if text.str.fullmatch(DATE_PATTERN).all():          # ISO dates or months (YYYY-MM[-DD])
        return f"{text.min()[:len('YYYY-MM')]} to {text.max()[:len('YYYY-MM')]}"
    return "text"


def file_guide(file_key):
    """One row per column of the raw file, with our name, meaning, use and computed facts."""
    raw, file_rename = RAW_FILES[file_key]
    rename = {**SHARED_RENAME, **file_rename}
    rows = []
    for original in raw.columns:
        key = f"{original} ({'leases' if file_key == 'lease_history' else 'listings'})" if original == "sRent" else original
        new, meaning, use = meanings[key]
        assert new.split(" ->")[0] == rename[original], f"dictionary and renaming disagree on {original}"
        rows.append({"original": original, "our name": new, "what it really contains": meaning, "how we use it": use,
                     "type": str(raw[original].dtype), "distinct": raw[original].nunique(),
                     "categories or range": summarise(raw[original], rename[original])})
    return pd.DataFrame(rows)


def show(file_key):
    """Read the CSV as is, rename its columns, show the first rows, then explain every column."""
    raw, file_rename = RAW_FILES[file_key]
    renamed = pd.read_csv(RAW_DIR / f"equinoxe_{file_key}.csv").rename(columns={**SHARED_RENAME, **file_rename})
    print(f"equinoxe_{file_key}.csv: {len(raw):,} rows, {raw.shape[1]} columns, all renamed. First rows with our names:")
    display(renamed.head(ROWS_PREVIEWED))
    guide = file_guide(file_key)
    print("What every column means:")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
        display(guide)

## 1. `equinoxe_listings.csv`: one row per apartment
The current state of each of the 1,061 apartments: what it is, where it is, and the rent it is advertised at today.
Note that `sRent` here is the **advertised** rent, so we call it `asking_rent_current`, not a signed rent.

In [ ]:
show("listings")

## 2. `equinoxe_lease_history.csv`: one row per lease
Every lease signed from 2017 to 2025. This is the file the forecast is built on. Here `sRent` is the rent **written on the
lease** (`rent_contract`), and `sRentEffective` is what was actually collected after discounts (`rent_effective`).
`sRenewal` = 1 means the same tenant renewed; 0 means a new tenant **or** the apartment's first lease.

In [ ]:
show("lease_history")

## 3. `equinoxe_concessions.csv`: one row per discount
Each discount given on a lease: free rent, parking, locker, appliances, a promo credit. `PromoPay` is described as monthly,
but it covers one month only: it is a one-off credit, so we rename it `one_time_promo_credit` (evidence in the main notebook, §2.6).

In [ ]:
show("concessions")

## 4. `equinoxe_asking_history.csv`: one row per apartment per month on the market
The rent advertised each month an apartment was for rent. It is not a signed rent, so we use it as context only.

In [ ]:
show("asking_history")

## Check: nothing left with a Yardi name
**What we do:** after renaming, list any column in the four tables that still has its original name. The answer should be
none.

In [ ]:
left_over = {name: [c for c in raw.columns if c not in {**SHARED_RENAME, **file_rename}]
             for name, (raw, file_rename) in RAW_FILES.items()}
print("Columns still carrying a Yardi name:", {k: v for k, v in left_over.items() if v} or "none")
print("Columns used in the analysis after renaming (leases):", ", ".join(leases.columns))